# Módulo 37: Operaciones con múltiples tablas

Autor: Juliho Castillo Colmenares

> Lección complementaria del ejercicio de este módulo: [enunciado](m37-operaciones-con-multiples-tablas-guia.ipynb) y [solución](ejercicios/). No es una tarea entregada del programa.

En una base relacional, la información de un mismo hecho se reparte entre varias tablas: el cliente en una, la persona en otra, la orden en una tercera. Para responder preguntas reales hay que **volver a juntarlas**. En esta lección veremos los distintos tipos de `JOIN`, cómo apilar resultados con `UNION` y cómo tratar los `NULL` que aparecen cuando una fila no tiene pareja.

1. Claves primarias y foráneas
2. `INNER JOIN`
3. `LEFT JOIN`
4. `RIGHT JOIN` y el orden de las tablas
5. `FULL OUTER JOIN`
6. `CROSS JOIN`
7. `UNION` frente a `UNION ALL`
8. `NULL`, `COALESCE` y `CASE`
9. Errores frecuentes
10. Ejercicio

## 1. Claves primarias y foráneas

La **clave primaria** identifica cada fila de una tabla. Una **clave foránea** es una columna que guarda la clave primaria de *otra* tabla y así crea la relación entre ambas.

| Tabla | Clave primaria | Clave foránea | Apunta a |
| --- | --- | --- | --- |
| `Person` | `BusinessEntityID` | | |
| `Customer` | `CustomerID` | `PersonID` | `Person.BusinessEntityID` |
| `SalesOrderHeader` | `SalesOrderID` | `CustomerID` | `Customer.CustomerID` |

En la base mínima que creamos hay casos incompletos a propósito: clientes que son tiendas (sin `PersonID`), un cliente sin órdenes y una orden cuyo cliente ya no existe. Son justo los casos en que los tipos de `JOIN` se distinguen.

In [1]:
import sqlite3

import pandas as pd

conn = sqlite3.connect(":memory:")
conn.executescript("""
CREATE TABLE Person (
    BusinessEntityID INTEGER PRIMARY KEY,
    FirstName TEXT, LastName TEXT
);
CREATE TABLE Customer (
    CustomerID INTEGER PRIMARY KEY,
    PersonID   INTEGER REFERENCES Person(BusinessEntityID),   -- NULL si el cliente es una tienda
    StoreID    INTEGER
);
CREATE TABLE SalesOrderHeader (
    SalesOrderID INTEGER PRIMARY KEY,
    CustomerID   INTEGER,          -- sin REFERENCES: admite una orden huérfana
    OrderDate    TEXT,
    TotalDue     REAL
);
INSERT INTO Person VALUES
 (1, 'Ken', 'Sánchez'), (2, 'Terri', 'Duffy'), (3, 'Gail', 'Erickson'),
 (4, 'Rob', 'Walters'), (5, 'Janice', 'Galvin');
INSERT INTO Customer VALUES
 (11000, 1, NULL), (11001, 2, NULL), (11002, 3, NULL),
 (11003, NULL, 292), (11004, NULL, 294), (11005, 4, NULL);
INSERT INTO SalesOrderHeader VALUES
 (43659, 11000, '2011-05-31', 2.5e4), (43660, 11001, '2011-06-01', 1.2e3),
 (43661, 11000, '2011-07-01', 3.4e3), (43662, 11003, '2011-07-15', 9.8e3),
 (43663, 11002, '2011-08-01', 4.4e2), (43664, 99999, '2011-08-03', 7.0e2);
""")


def q(sql):
    """Ejecuta una consulta y devuelve el resultado como DataFrame."""
    return pd.read_sql_query(sql, conn)


display(q("SELECT * FROM Customer"))
q("SELECT * FROM SalesOrderHeader")

,CustomerID,PersonID,StoreID
0,11000,1.0,NaN
1,11001,2.0,NaN
2,11002,3.0,NaN
3,11003,NaN,292.0
4,11004,NaN,294.0
5,11005,4.0,NaN


,SalesOrderID,CustomerID,OrderDate,TotalDue
0,43659,11000,2011-05-31,25000.0
1,43660,11001,2011-06-01,1200.0
2,43661,11000,2011-07-01,3400.0
3,43662,11003,2011-07-15,9800.0
4,43663,11002,2011-08-01,440.0
5,43664,99999,2011-08-03,700.0


## 2. `INNER JOIN`

`INNER JOIN` devuelve solo las combinaciones de filas que cumplen la condición `ON` en **ambas** tablas. Las filas sin pareja desaparecen.

Los **alias** de tabla (`c`, `p`) acortan la consulta y evitan ambigüedad cuando dos tablas tienen columnas con el mismo nombre.

In [2]:
q("""
SELECT c.CustomerID, p.FirstName, p.LastName
FROM Customer c
INNER JOIN Person p ON c.PersonID = p.BusinessEntityID
""")

,CustomerID,FirstName,LastName
0,11000,Ken,Sánchez
1,11001,Terri,Duffy
2,11002,Gail,Erickson
3,11005,Rob,Walters


Las tiendas 11003 y 11004 no aparecen: su `PersonID` es `NULL` y no coincide con ninguna persona. Tampoco aparece Janice (persona 5), que no es cliente.

## 3. `LEFT JOIN`

`LEFT JOIN` conserva **todas las filas de la tabla de la izquierda** (la que va en `FROM`) y les añade las columnas de la derecha cuando hay coincidencia. Si no la hay, esas columnas quedan en `NULL`.

In [3]:
q("""
SELECT c.CustomerID, c.StoreID, p.FirstName, p.LastName
FROM Customer c
LEFT JOIN Person p ON c.PersonID = p.BusinessEntityID
""")

,CustomerID,StoreID,FirstName,LastName
0,11000,NaN,Ken,Sánchez
1,11001,NaN,Terri,Duffy
2,11002,NaN,Gail,Erickson
3,11003,292.0,None,None
4,11004,294.0,None,None
5,11005,NaN,Rob,Walters


Al pasar el resultado a pandas, los `NULL` se muestran como `None` en columnas de texto y como `NaN` en columnas numéricas (que por eso aparecen con decimales, como `StoreID`).

Un uso frecuente es buscar filas **sin pareja**: después del `LEFT JOIN`, se filtran las que tienen `NULL` en la clave de la derecha. Así se obtienen los clientes que nunca han comprado:

In [4]:
q("""
SELECT c.CustomerID
FROM Customer c
LEFT JOIN SalesOrderHeader soh ON soh.CustomerID = c.CustomerID
WHERE soh.SalesOrderID IS NULL
""")

,CustomerID
0,11004
1,11005


## 4. `RIGHT JOIN` y el orden de las tablas

`RIGHT JOIN` es el espejo de `LEFT JOIN`: conserva todas las filas de la tabla de la **derecha** (la que va después de `RIGHT JOIN`). En los dos casos, el orden en que se escriben las tablas decide qué filas se conservan:

| Si se quieren todas las filas de `A` | Se escribe |
| --- | --- |
| Con `LEFT JOIN` | `FROM A LEFT JOIN B ON ...` (A a la izquierda) |
| Con `RIGHT JOIN` | `FROM B RIGHT JOIN A ON ...` (A a la derecha) |

Por eso `A LEFT JOIN B` y `B RIGHT JOIN A` dan el mismo resultado. Veamos las dos posiciones posibles con órdenes y clientes:

In [5]:
# Órdenes a la derecha: se conservan TODAS las órdenes, incluso la huérfana 43664
display(q("""
SELECT soh.SalesOrderID, soh.CustomerID AS ClienteEnOrden, c.CustomerID
FROM Customer c
RIGHT JOIN SalesOrderHeader soh ON soh.CustomerID = c.CustomerID
"""))

# Clientes a la derecha: se conservan TODOS los clientes, incluso los que no compraron
q("""
SELECT soh.SalesOrderID, c.CustomerID
FROM SalesOrderHeader soh
RIGHT JOIN Customer c ON soh.CustomerID = c.CustomerID
""")

,SalesOrderID,ClienteEnOrden,CustomerID
0,43659,11000,11000.0
1,43661,11000,11000.0
2,43660,11001,11001.0
3,43663,11002,11002.0
4,43662,11003,11003.0
5,43664,99999,NaN


,SalesOrderID,CustomerID
0,43659.0,11000
1,43660.0,11001
2,43661.0,11000
3,43662.0,11003
4,43663.0,11002
5,NaN,11004
6,NaN,11005


Las dos consultas usan `RIGHT JOIN` y la misma condición, pero responden preguntas distintas. La primera muestra la orden 43664 con `CustomerID` nulo (su cliente no existe); la segunda muestra los clientes 11004 y 11005 sin orden y omite la orden huérfana.

## 5. `FULL OUTER JOIN`

`FULL OUTER JOIN` conserva las filas de **ambas** tablas: las que coinciden se combinan, y las que no tienen pareja aparecen con `NULL` en las columnas de la otra tabla. Es la unión de un `LEFT JOIN` y un `RIGHT JOIN`.

In [6]:
conn.executescript("""
CREATE TABLE Product (ProductID INTEGER PRIMARY KEY, Name TEXT);
CREATE TABLE ProductReview (ProductReviewID INTEGER PRIMARY KEY, ProductID INTEGER, Comments TEXT);
INSERT INTO Product VALUES (1, 'Mountain-100'), (2, 'Road-150'), (3, 'Sport-100 Helmet'), (4, 'Water Bottle');
INSERT INTO ProductReview VALUES
 (1, 1, 'Muy resistente'), (2, 3, 'Ligero y cómodo'), (3, 3, 'Buen precio'),
 (4, 99, 'Producto descontinuado');
""")

q("""
SELECT p.ProductID, p.Name, r.ProductID AS ProductoReseñado, r.Comments
FROM Product p
FULL OUTER JOIN ProductReview r ON p.ProductID = r.ProductID
""")

,ProductID,Name,ProductoReseñado,Comments
0,1.0,Mountain-100,1.0,Muy resistente
1,2.0,Road-150,NaN,None
2,3.0,Sport-100 Helmet,3.0,Ligero y cómodo
3,3.0,Sport-100 Helmet,3.0,Buen precio
4,4.0,Water Bottle,NaN,None
5,NaN,None,99.0,Producto descontinuado


Los productos 2 y 4 no tienen reseñas, y la reseña del producto 99 no tiene producto. Las tres situaciones aparecen en el resultado.

## 6. `CROSS JOIN`

`CROSS JOIN` no lleva condición: combina **cada fila de una tabla con cada fila de la otra** (producto cartesiano). Con $n$ y $m$ filas, el resultado tiene $n \times m$ filas. Sirve para generar todas las combinaciones posibles, por ejemplo cada producto con cada categoría o cada tienda con cada mes.

In [7]:
conn.executescript("""
CREATE TABLE ProductCategory (ProductCategoryID INTEGER PRIMARY KEY, Name TEXT);
INSERT INTO ProductCategory VALUES (1, 'Bikes'), (2, 'Accessories'), (3, 'Clothing');
""")

combinaciones = q("""
SELECT p.Name AS Producto, pc.Name AS Categoria
FROM Product p
CROSS JOIN ProductCategory pc
""")
print("Filas:", len(combinaciones), "= 4 productos x 3 categorías")
combinaciones.head(6)

Filas: 12 = 4 productos x 3 categorías


,Producto,Categoria
0,Mountain-100,Bikes
1,Mountain-100,Accessories
2,Mountain-100,Clothing
3,Road-150,Bikes
4,Road-150,Accessories
5,Road-150,Clothing


## 7. `UNION` frente a `UNION ALL`

Los `JOIN` agregan **columnas**; `UNION` agrega **filas**: apila el resultado de dos consultas que tienen el mismo número de columnas, en el mismo orden y con tipos compatibles.

| Operador | Duplicados | Costo |
| --- | --- | --- |
| `UNION` | Los elimina | Mayor: debe comparar todas las filas |
| `UNION ALL` | Los conserva | Menor |

In [8]:
conn.executescript("""
CREATE TABLE ProductModel (ProductModelID INTEGER PRIMARY KEY, Name TEXT);
INSERT INTO ProductModel VALUES (1, 'Mountain-100'), (2, 'Road-150'), (3, 'Touring-1000');
""")

union = q("SELECT Name FROM Product UNION SELECT Name FROM ProductModel ORDER BY Name")
union_all = q("SELECT Name FROM Product UNION ALL SELECT Name FROM ProductModel ORDER BY Name")
print("UNION:", len(union), "filas | UNION ALL:", len(union_all), "filas")
union_all

UNION: 5 filas | UNION ALL: 7 filas


,Name
0,Mountain-100
1,Mountain-100
2,Road-150
3,Road-150
4,Sport-100 Helmet
5,Touring-1000
6,Water Bottle


`Mountain-100` y `Road-150` están en las dos tablas: `UNION ALL` las repite y `UNION` las deja una vez. El `ORDER BY` se escribe una sola vez, al final, y ordena el resultado combinado.

## 8. `NULL`, `COALESCE` y `CASE`

Para reemplazar un `NULL` por un valor legible hay tres funciones:

| Función | Dónde existe | Argumentos |
| --- | --- | --- |
| `COALESCE(a, b, c, ...)` | Estándar: SQLite, SQL Server y casi todos | Devuelve el primero no nulo |
| `IFNULL(a, b)` | SQLite, MySQL | Exactamente dos |
| `ISNULL(a, b)` | SQL Server | Exactamente dos; el resultado toma el tipo de `a` |

`CASE` combinado con `LIKE` clasifica filas según patrones de texto. Un detalle: si la columna es `NULL`, `x LIKE '%...%'` no es verdadero, así que la fila cae en el `ELSE`.

In [9]:
conn.executescript("""
CREATE TABLE Employee (BusinessEntityID INTEGER PRIMARY KEY, JobTitle TEXT);
INSERT INTO Employee VALUES
 (1, 'Chief Executive Officer'), (2, 'Engineering Manager'), (3, 'Design Engineer'),
 (4, NULL), (5, 'Production Manager - WC10'), (6, 'Production Technician');
CREATE TABLE SalesPerson (BusinessEntityID INTEGER PRIMARY KEY, SalesQuota REAL);
INSERT INTO SalesPerson VALUES (274, NULL), (275, 300000), (276, 250000), (285, NULL);
""")

display(q("""
SELECT BusinessEntityID,
       COALESCE(JobTitle, 'Sin puesto') AS Puesto,
       CASE
           WHEN JobTitle LIKE '%Manager%' THEN 'Gerente'
           WHEN JobTitle IS NULL THEN 'Sin dato'
           ELSE 'Otro'
       END AS Tipo
FROM Employee
"""))

q("SELECT BusinessEntityID, IFNULL(SalesQuota, 'Sin cuota') AS Cuota FROM SalesPerson")

,BusinessEntityID,Puesto,Tipo
0,1,Chief Executive Officer,Otro
1,2,Engineering Manager,Gerente
2,3,Design Engineer,Otro
3,4,Sin puesto,Sin dato
4,5,Production Manager - WC10,Gerente
5,6,Production Technician,Otro


,BusinessEntityID,Cuota
0,274,Sin cuota
1,275,300000.0
2,276,250000.0
3,285,Sin cuota


En T-SQL, la última consulta necesita convertir la cuota a texto, porque `ISNULL` intenta convertir `'Sin cuota'` al tipo numérico de `SalesQuota` y falla:

```sql
SELECT BusinessEntityID,
       ISNULL(CAST(SalesQuota AS VARCHAR(20)), 'Sin cuota') AS Cuota
FROM Sales.SalesPerson;
```

## 9. Errores frecuentes

**Poner la tabla equivocada a cada lado.** La consulta no da error: simplemente conserva las filas de la otra tabla. Antes de escribir un `LEFT` o `RIGHT JOIN` conviene decir en voz alta "quiero todas las filas de ___" y colocar esa tabla a la izquierda del `LEFT` o a la derecha del `RIGHT`.

**Filtrar la tabla opcional en `WHERE`.** Una condición sobre la tabla de la derecha en el `WHERE` descarta las filas donde esa tabla es `NULL` y convierte el `LEFT JOIN` en un `INNER JOIN`. Si el filtro debe aplicarse antes de combinar, va en el `ON`.

In [10]:
# Mal: el filtro en WHERE elimina a los clientes sin órdenes en julio
display(q("""
SELECT c.CustomerID, soh.SalesOrderID
FROM Customer c
LEFT JOIN SalesOrderHeader soh ON soh.CustomerID = c.CustomerID
WHERE soh.OrderDate >= '2011-07-01'
"""))

# Bien: el filtro en ON conserva a todos los clientes
q("""
SELECT c.CustomerID, soh.SalesOrderID
FROM Customer c
LEFT JOIN SalesOrderHeader soh
       ON soh.CustomerID = c.CustomerID AND soh.OrderDate >= '2011-07-01'
""")

,CustomerID,SalesOrderID
0,11000,43661
1,11003,43662
2,11002,43663


,CustomerID,SalesOrderID
0,11000,43661.0
1,11001,NaN
2,11002,43663.0
3,11003,43662.0
4,11004,NaN
5,11005,NaN


**Olvidar que un `JOIN` multiplica filas.** Si un cliente tiene dos órdenes, aparece dos veces tras el `JOIN`. Contar clientes con `COUNT(*)` después de unir con órdenes da un número inflado; se usa `COUNT(DISTINCT c.CustomerID)`.

In [11]:
q("""
SELECT COUNT(*) AS filas, COUNT(DISTINCT c.CustomerID) AS clientes_distintos
FROM Customer c
INNER JOIN SalesOrderHeader soh ON soh.CustomerID = c.CustomerID
""")

,filas,clientes_distintos
0,5,4


**Comparar con `NULL` en la condición del `JOIN`.** `c.PersonID = p.BusinessEntityID` nunca es verdadero si `PersonID` es `NULL`; por eso las tiendas no se unen con ninguna persona. No es un error del motor, es la semántica de `NULL`.

## 10. Ejercicio

Una empresa organiza a sus vendedores por territorio:

```python
territorios = [(1, "Northwest"), (2, "Northeast"), (3, "Central"), (4, "Canada")]
vendedores = [  # (VendedorID, Nombre, TerritoryID, Cuota)
    (274, "Stephen Jiang", None, None),
    (275, "Michael Blythe", 2, 300000),
    (276, "Linda Mitchell", 1, 250000),
    (277, "Jillian Carson", 3, 250000),
    (278, "Garrett Vargas", 1, None),
]
```

1. Crea las tablas `Territory` y `Vendedor` e inserta los datos.
2. Lista cada vendedor con el nombre de su territorio, conservando a los que no tienen territorio. Muestra `'Sin territorio'` en lugar de `NULL`.
3. Lista **todos los territorios** con sus vendedores usando `RIGHT JOIN`, y di cuál no tiene vendedores.
4. Muestra cada vendedor con una columna `Estado` que valga `'Sin cuota'` si la cuota es nula, `'Alta'` si es al menos 300000 y `'Normal'` en otro caso.

### Solución

In [12]:
territorios = [(1, "Northwest"), (2, "Northeast"), (3, "Central"), (4, "Canada")]
vendedores = [
    (274, "Stephen Jiang", None, None),
    (275, "Michael Blythe", 2, 300000),
    (276, "Linda Mitchell", 1, 250000),
    (277, "Jillian Carson", 3, 250000),
    (278, "Garrett Vargas", 1, None),
]
conn.execute("CREATE TABLE Territory (TerritoryID INTEGER PRIMARY KEY, Name TEXT)")
conn.execute("CREATE TABLE Vendedor (VendedorID INTEGER PRIMARY KEY, Nombre TEXT, TerritoryID INTEGER, Cuota REAL)")
conn.executemany("INSERT INTO Territory VALUES (?, ?)", territorios)
conn.executemany("INSERT INTO Vendedor VALUES (?, ?, ?, ?)", vendedores)

q("""
SELECT v.Nombre, COALESCE(t.Name, 'Sin territorio') AS Territorio
FROM Vendedor v
LEFT JOIN Territory t ON v.TerritoryID = t.TerritoryID
""")

,Nombre,Territorio
0,Stephen Jiang,Sin territorio
1,Michael Blythe,Northeast
2,Linda Mitchell,Northwest
3,Jillian Carson,Central
4,Garrett Vargas,Northwest


In [13]:
# Todos los territorios: Territory va a la DERECHA del RIGHT JOIN
q("""
SELECT t.Name AS Territorio, v.Nombre
FROM Vendedor v
RIGHT JOIN Territory t ON v.TerritoryID = t.TerritoryID
ORDER BY t.TerritoryID
""")

,Territorio,Nombre
0,Northwest,Linda Mitchell
1,Northwest,Garrett Vargas
2,Northeast,Michael Blythe
3,Central,Jillian Carson
4,Canada,None


Canada es el único territorio sin vendedores: aparece con `Nombre` nulo.

In [14]:
q("""
SELECT Nombre,
       CASE
           WHEN Cuota IS NULL THEN 'Sin cuota'
           WHEN Cuota >= 300000 THEN 'Alta'
           ELSE 'Normal'
       END AS Estado
FROM Vendedor
""")

,Nombre,Estado
0,Stephen Jiang,Sin cuota
1,Michael Blythe,Alta
2,Linda Mitchell,Normal
3,Jillian Carson,Normal
4,Garrett Vargas,Sin cuota


In [15]:
conn.close()

## Resumen

- La clave foránea guarda la clave primaria de otra tabla y es la base de los `JOIN`.
- `INNER JOIN` conserva solo las coincidencias; `LEFT JOIN` todas las filas de la izquierda; `RIGHT JOIN` todas las de la derecha; `FULL OUTER JOIN` todas las de ambas.
- El orden de las tablas decide qué filas se conservan: para quedarse con todo `A`, se escribe `A LEFT JOIN B` o `B RIGHT JOIN A`.
- `CROSS JOIN` genera todas las combinaciones ($n \times m$ filas); `UNION` apila filas sin duplicados y `UNION ALL` con ellos.
- `COALESCE` es estándar; `IFNULL` (SQLite) e `ISNULL` (T-SQL) son sus versiones de dos argumentos.

En el **Módulo 38** se agrupan filas con `GROUP BY` y `HAVING`, y se calculan funciones de ventana con `OVER` y `PARTITION BY`.